In [1]:
from scholar_rank.ingest.fetch_data import get_manifest, list_local_and_remote_shards, fetch_shard, extract_compact
from scholar_rank.utils import PROJECT_ROOT

# Fetching work entities from OpenAlex S3 storage
UPSTREAM_MANIFEST_PATH: Path("data/parquet/works/manifest.json")
RAW_DATA_PATH: Path = PROJECT_ROOT/"data"/"openalex"
PROCESSED_DATA_PATH: Path = PROJECT_ROOT/"data"/"processed"

manifest_path = RAW_DATA_PATH/"works"/"manifest.json"



data = list_local_and_remote_shards(manifest_path,RAW_DATA_PATH)
print(data[0])
print(data[1])

['works/updated_date=2016-06-24/part_0000.parquet', 'works/updated_date=2016-07-22/part_0000.parquet', 'works/updated_date=2016-08-23/part_0000.parquet', 'works/updated_date=2016-09-16/part_0000.parquet', 'works/updated_date=2016-09-23/part_0000.parquet', 'works/updated_date=2016-09-30/part_0000.parquet', 'works/updated_date=2016-10-07/part_0000.parquet', 'works/updated_date=2016-10-14/part_0000.parquet', 'works/updated_date=2016-10-21/part_0000.parquet', 'works/updated_date=2016-10-28/part_0000.parquet', 'works/updated_date=2016-11-04/part_0000.parquet', 'works/updated_date=2016-11-11/part_0000.parquet', 'works/updated_date=2016-11-30/part_0000.parquet', 'works/updated_date=2016-12-08/part_0000.parquet', 'works/updated_date=2016-12-16/part_0000.parquet', 'works/updated_date=2017-01-06/part_0000.parquet', 'works/updated_date=2017-01-13/part_0000.parquet', 'works/updated_date=2017-01-26/part_0000.parquet', 'works/updated_date=2017-02-03/part_0000.parquet', 'works/updated_date=2017-02-10

In [ ]:
from scholar_rank.ingest.fetch_data import show_shard

# get_manifest(UPSTREAM_MANIFEST_PATH, manifest_path)

sample_parquet = "works/updated_date=2016-10-28/part_0000.parquet"

show_shard(RAW_DATA_PATH/sample_parquet)

┌──────────────────────────────────┬────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬──────────────────┬──────────────────┬──────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [3]:
extract_compact(RAW_DATA_PATH/sample_parquet, PROCESSED_DATA_PATH/sample_parquet)

show_shard(PROCESSED_DATA_PATH/sample_parquet)

┌─────────────┬────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [4]:
from pathlib import Path

UPSTREAM_PREFIX = Path("data/parquet")

sample_upstream_shard = 'works/updated_date=2026-02-11/part_0000.parquet'
fetch_shard(UPSTREAM_PREFIX/sample_upstream_shard, RAW_DATA_PATH/sample_upstream_shard)

2026-07-15 12:51:19,559 [INFO] scholar_rank.ingest.fetch_data: Fetching data/parquet/works/updated_date=2026-02-11/part_0000.parquet successfully.


In [23]:
# Checking null-rate for all record in current local repo

import duckdb

columns = [
    "id",
    "doi",
    "title",
    "authorships",
    "abstract_inverted_index",
    "type",
    "language",
    "primary_location",
    "publication_year",
    "publication_date",
    "referenced_works",
    "referenced_works_count",
    "cited_by_count",
    "topics"
]
lst = "[" + ", ".join(f"'{c}'" for c in columns) + "]"

local_shard_list, remote_shard_list = list_local_and_remote_shards(manifest_path, RAW_DATA_PATH)

running_total = 0
running_null = dict.fromkeys(columns,0)

db = duckdb.connect()

print(f"{running_total} {running_null}")

for i in range(0,len(local_shard_list)):
    local = db.cursor()

    shard_path = local_shard_list[i]
    print(f"Loading file {i+1} out of {len(local_shard_list)}: {shard_path}", flush=True)
    rel = local.read_parquet(str(RAW_DATA_PATH/shard_path)).aggregate(
        f'count(*) AS total, count(*) - count(COLUMNS({lst})) AS "\\0"'
    )
    row = rel.fetchone()
    running_total += row[0]
    for name, val in zip(rel.columns[1:], row[1:]):
        running_null[name] += val

with open("null_analysis.txt", "w", encoding="utf-8") as file:
    file.write(f"Total entries: {running_total}.\n")
    file.write(f"Columns labels:\n{columns}\n")
    file.write(f"Raw NULL count by column:\n{running_null}\n")
    file.write(f"Percent NULL by column\n{ {key: (float(value)/running_total) for key, value in running_null.items()} }\n")

0 {'id': 0, 'doi': 0, 'title': 0, 'authorships': 0, 'abstract_inverted_index': 0, 'type': 0, 'language': 0, 'primary_location': 0, 'publication_year': 0, 'publication_date': 0, 'referenced_works': 0, 'referenced_works_count': 0, 'cited_by_count': 0, 'topics': 0}
Loading file 1 out of 1857: works/updated_date=2016-06-24/part_0000.parquet
Loading file 2 out of 1857: works/updated_date=2016-07-22/part_0000.parquet
Loading file 3 out of 1857: works/updated_date=2016-08-23/part_0000.parquet
Loading file 4 out of 1857: works/updated_date=2016-09-16/part_0000.parquet
Loading file 5 out of 1857: works/updated_date=2016-09-23/part_0000.parquet
Loading file 6 out of 1857: works/updated_date=2016-09-30/part_0000.parquet
Loading file 7 out of 1857: works/updated_date=2016-10-07/part_0000.parquet
Loading file 8 out of 1857: works/updated_date=2016-10-14/part_0000.parquet
Loading file 9 out of 1857: works/updated_date=2016-10-21/part_0000.parquet
Loading file 10 out of 1857: works/updated_date=2016-